In [ ]:
```python
import asyncio
import aiohttp


async def download_first(urls: list[str]) -> bytes:
    async def fetch(session: aiohttp.ClientSession, url: str) -> bytes:
        async with session.get(url) as response:
            response.raise_for_status()
            return await response.read()

    async with aiohttp.ClientSession() as session:
        tasks = [asyncio.create_task(fetch(session, url)) for url in urls]

        try:
            done, pending = await asyncio.wait(
                tasks,
                timeout=5,
                return_when=asyncio.FIRST_COMPLETED,
            )

            if not done:
                raise TimeoutError("下载超时")

            for task in done:
                try:
                    result = task.result()
                except Exception:
                    continue
                else:
                    for pending_task in pending:
                        pending_task.cancel()
                    await asyncio.gather(*pending, return_exceptions=True)
                    return result

            raise RuntimeError("所有下载均失败")
        finally:
            for task in tasks:
                if not task.done():
                    task.cancel()
            await asyncio.gather(*tasks, return_exceptions=True)


# 调用示例
urls = [
    "https://example.com/a",
    "https://example.com/b",
    "https://example.com/c",
]

content = asyncio.run(download_first(urls))
print(content[:100])
```

需要安装依赖：`pip install aiohttp`。